In [0]:
base_path = "abfss://adlscontainer1@parishah16.dfs.core.windows.net/"
silver_path = f"{base_path}silver/ecommerce/"
gold_path   = f"{base_path}gold/ecommerce/"

In [0]:
orders_df = spark.read.parquet(f"{base_path}silver/ecommerce/orders")
order_items_df = spark.read.parquet(f"{base_path}silver/ecommerce/order_items")
payments_df = spark.read.parquet(f"{base_path}silver/ecommerce/olist_order_payments_dataset")
customers_df = spark.read.parquet(f"{base_path}silver/ecommerce/customers")
products_df = spark.read.parquet(f"{base_path}silver/ecommerce/products")
sellers_df = spark.read.parquet(f"{base_path}silver/ecommerce/sellers")
reviews_df = spark.read.parquet(f"{base_path}silver/ecommerce/reviews")

In [0]:
from pyspark.sql.functions import col
dim_customers = customers_df.select(
    "customer_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix",
    "customer_region",   
    "customer_location"
).dropDuplicates()

dim_customers.write.mode("overwrite").parquet(f"{gold_path}dim_customers")

In [0]:
dim_products = products_df.select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_volume_cm3",
    "size_category",
    "weight_category"
).dropDuplicates()

dim_products.write.mode("overwrite").parquet(f"{gold_path}dim_products")

In [0]:
dim_sellers = sellers_df.select(
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix",
    "seller_region"
).dropDuplicates()

dim_sellers.write.mode("overwrite").parquet(f"{gold_path}dim_sellers")

In [0]:
from pyspark.sql.functions import *

dim_date = orders_df.select("order_purchase_timestamp") \
    .withColumn("date", to_date("order_purchase_timestamp")) \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("week", weekofyear("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("day_name", date_format("date","EEEE")) \
    .withColumn("is_weekend",
        when(col("day_name").isin("Saturday","Sunday"),1).otherwise(0)
    ).dropDuplicates()

dim_date.write.mode("overwrite").parquet(f"{gold_path}dim_date")


In [0]:
from pyspark.sql.functions import sum
payments_agg = payments_df.groupBy("order_id") \
    .agg(sum("payment_value").alias("total_payment"))

In [0]:
orders_df = orders_df.drop("load_time")
order_items_df = order_items_df.drop("load_time")
payments_df = payments_df.drop("load_time")
customers_df = customers_df.drop("load_time")
products_df = products_df.drop("load_time")
sellers_df = sellers_df.drop("load_time")

In [0]:
customers_df = customers_df.withColumnRenamed("region", "customer_region")
sellers_df   = sellers_df.withColumnRenamed("region", "seller_region")

In [0]:
fact_df = orders_df.alias("o") \
    .join(order_items_df.alias("oi"), "order_id", "inner") \
    .join(payments_agg.alias("p"), "order_id", "left") \
    .join(dim_customers.alias("c"), "customer_id", "left") \
    .join(dim_products.alias("pr"), "product_id", "left") \
    .join(dim_sellers.alias("s"), "seller_id", "left")

In [0]:
fact_df = fact_df \
    .withColumn("total_revenue", col("price") + col("freight_value")) \
    .withColumn("profit_estimate", col("price") * 0.2) \
    .withColumn("delivery_days",
        datediff("order_delivered_customer_date","order_purchase_timestamp")) \
    .withColumn("is_late_delivery",
        when(col("delivery_days") > 7,1).otherwise(0)) \
    .withColumn("order_year", year("order_purchase_timestamp")) \
    .withColumn("order_month", month("order_purchase_timestamp"))

In [0]:
fact_df = fact_df.withColumn("gold_load_time", current_timestamp())

In [0]:
fact_df.write.mode("overwrite") \
    .partitionBy("order_year","order_month") \
    .parquet(f"{gold_path}fact_sales")

In [0]:
fact_df.count()

112650

In [0]:
fact_df.groupBy("order_id").count().orderBy("count", ascending=False).show()

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|8272b63d03f5f79c5...|   21|
|1b15974a0141d54e3...|   20|
|ab14fdcfbe524636d...|   20|
|428a2f660dc84138d...|   15|
|9ef13efd6949e4573...|   15|
|9bdc4d4c71aa1de46...|   14|
|73c8ab38f07dc9438...|   14|
|37ee401157a3a0b28...|   13|
|3a213fcdfe7d98be7...|   12|
|af822dacd6f5cff73...|   12|
|c05d6a79e55da72ca...|   12|
|2c2a19b5703863c90...|   12|
|637617b3ffe9e2f7a...|   12|
|7f2c22c54cbae5509...|   11|
|71dab1155600756af...|   11|
|5a3b1c29a49756e75...|   11|
|6c355e2913545fa6f...|   11|
|a483ffe0ce133740a...|   10|
|9f5054bd9a3c71702...|   10|
|9aec4e1ae90b23c7b...|   10|
+--------------------+-----+
only showing top 20 rows


In [0]:
fact_df.selectExpr("sum(total_revenue)").show()

+--------------------+
|  sum(total_revenue)|
+--------------------+
|1.5843553240000103E7|
+--------------------+



In [0]:
order_items_df.groupBy("order_id").count().orderBy("count", ascending=False).show()

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|8272b63d03f5f79c5...|   21|
|ab14fdcfbe524636d...|   20|
|1b15974a0141d54e3...|   20|
|428a2f660dc84138d...|   15|
|9ef13efd6949e4573...|   15|
|73c8ab38f07dc9438...|   14|
|9bdc4d4c71aa1de46...|   14|
|37ee401157a3a0b28...|   13|
|c05d6a79e55da72ca...|   12|
|3a213fcdfe7d98be7...|   12|
|af822dacd6f5cff73...|   12|
|637617b3ffe9e2f7a...|   12|
|2c2a19b5703863c90...|   12|
|7f2c22c54cbae5509...|   11|
|5a3b1c29a49756e75...|   11|
|6c355e2913545fa6f...|   11|
|71dab1155600756af...|   11|
|a483ffe0ce133740a...|   10|
|9aec4e1ae90b23c7b...|   10|
|f80549a97eb203e15...|   10|
+--------------------+-----+
only showing top 20 rows
